# 04. 오분류 분석
담당: 데이터 A · 검증셋에서 모델이 틀린 사례를 모아 원인을 찾고, 다음 실험(정제·증강·모델)을 제안합니다.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.utils import load_class_map, load_config
from src.visualize import set_korean_font

set_korean_font()
pd.set_option("display.max_columns", 50)

CONFIG = "configs/base.yaml"  # 샘플 데이터로 연습: "configs/sample_yolo.yaml"
cfg = load_config(CONFIG)
d = cfg["data"]

In [2]:
EXP = "sample_yolo"  # 분석할 실험 이름

from src.interfaces import OUTPUT_DIR
from src.metrics import confusion_pairs, match_predictions
from src.train import val_targets
from src.utils import load_json, predictions_from_json

exp_cfg = load_config(OUTPUT_DIR / EXP / "config.yaml")
paths, targets = val_targets(exp_cfg)
preds = predictions_from_json(load_json(OUTPUT_DIR / EXP / "preds_val.json"))
rows = pd.DataFrame(match_predictions(preds, targets, iou_thr=0.75, conf_thr=0.3))
rows["result"].value_counts()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/pool/Desktop/Beginner_Project/YacssoK/outputs/sample_yolo/config.yaml'

## 자주 헷갈리는 클래스 쌍
같은 쌍이 반복되면 해상도 증가(D6)나 2단계 분류(M4) 실험의 근거가 됩니다.

In [ ]:
class_map = load_class_map(exp_cfg["data"]["class_map"])
names = [c["name"] for c in class_map]
pd.DataFrame(
    [(names[g], names[p], n) for g, p, n in confusion_pairs(rows.to_dict("records"))],
    columns=["정답", "예측", "횟수"],
)

## 틀린 이미지 보기

In [ ]:
from src.dataset import read_image_rgb
from src.visualize import show_prediction

bad_ids = rows.loc[rows["result"] != "correct", "image_id"].unique()[:5]
for p, t, path in zip(preds, targets, paths):
    if t["image_id"] in bad_ids:
        show_prediction(read_image_rgb(path), p, t["boxes"], t["labels"], title=path.name)

## 정리
- 놓친 알약의 공통점:
- 헷갈린 쌍의 공통점:
- 제안하는 다음 실험: